# vec2text Colab Smoke Test and Small Eval

This notebook is designed for Google Colab and can be uploaded directly without pushing your local repo first.

It covers:
- environment setup
- loading HF pretrained models for inference
- `gtr-base` smoke test
- LMI smoke test
- small-scale eval with inline Python code

Before running:
- switch Colab runtime to GPU
- if you want to use your own fork later, change `REPO_URL`


## 1. Check GPU

In Colab: `Runtime` -> `Change runtime type` -> choose `T4 GPU` or another GPU.

In [ ]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu only')


## 2. Clone the repository and install dependencies

Default uses the public upstream repo so you do not need to push your own fork first.

In [ ]:
REPO_URL = 'https://github.com/jxmorris12/vec2text.git'
REPO_DIR = 'vec2text'


In [ ]:
!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"


In [ ]:
%cd /content/vec2text
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install -e .


## 3. Runtime setup

In [ ]:
import os

os.environ['VEC2TEXT_CACHE'] = '/content/vec2text_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.makedirs('/content/vec2text_cache', exist_ok=True)
os.makedirs('/content/vec2text/results', exist_ok=True)
print('VEC2TEXT_CACHE =', os.environ['VEC2TEXT_CACHE'])


## 4. Colab compatibility patches

The upstream repo currently needs two small runtime patches on Colab:
- add `import platform` in `experiments.py`
- disable `low_cpu_mem_usage` in `model_utils.py`
- disable `low_cpu_mem_usage` in `corrector_encoder.py`


In [ ]:
from pathlib import Path

# Patch 1: fix missing import in experiments.py
path1 = Path('/content/vec2text/vec2text/experiments.py')
text1 = path1.read_text(encoding='utf-8')
if 'import platform' not in text1:
    text1 = text1.replace('import os\n', 'import os\nimport platform\n', 1)
    path1.write_text(text1, encoding='utf-8')

# Patch 2: avoid meta-device loading issue in model_utils.py
path2 = Path('/content/vec2text/vec2text/models/model_utils.py')
text2 = path2.read_text(encoding='utf-8')
text2 = text2.replace('"low_cpu_mem_usage": True,', '"low_cpu_mem_usage": False,')
path2.write_text(text2, encoding='utf-8')

# Patch 3: avoid the same meta-device issue in corrector_encoder.py
path3 = Path('/content/vec2text/vec2text/models/corrector_encoder.py')
text3 = path3.read_text(encoding='utf-8')
old3 = """encoder_decoder = transformers.AutoModelForSeq2SeqLM.from_pretrained(\n            config.model_name_or_path\n        )"""
new3 = """encoder_decoder = transformers.AutoModelForSeq2SeqLM.from_pretrained(\n            config.model_name_or_path,\n            low_cpu_mem_usage=False,\n        )"""
if old3 in text3:
    text3 = text3.replace(old3, new3, 1)
    path3.write_text(text3, encoding='utf-8')

print('patched experiments.py, model_utils.py, and corrector_encoder.py')


## 4. Import the library

In [ ]:
import vec2text

print('vec2text imported successfully')


## 5. `gtr-base` smoke test

In [ ]:
corrector = vec2text.load_pretrained_corrector('gtr-base')
print('Loaded pretrained corrector successfully.')
print(type(corrector).__name__)


In [ ]:
texts = [
    'Jack Morris is a PhD student at Cornell Tech in New York City',
    'It was the best of times, it was the worst of times'
]

outputs = vec2text.invert_strings(
    texts,
    corrector=corrector,
    num_steps=5,
)

for i, text in enumerate(outputs):
    print(f'[{i}] {text}')


## 6. LMI smoke test

This is heavier than `gtr-base`. If it fails, common causes are GPU RAM limits, HF auth requirements, or transient download failures.

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

MODEL_NAME = 'jxm/t5-base__llama-7b__one-million-instructions__emb'
CORRECTOR_NAME = 'jxm/t5-base___llama-7b___one-million-instructions__correct'

inversion_model = vec2text.models.InversionFromLogitsEmbModel.from_pretrained(MODEL_NAME).to(device)
corrector_model = vec2text.models.CorrectorEncoderFromLogitsModel.from_pretrained(CORRECTOR_NAME).to(device)
lmi_corrector = vec2text.load_corrector(inversion_model, corrector_model)
print('Loaded inversion model and corrector successfully.')
print(type(lmi_corrector).__name__)


## 7. Small-scale eval

This notebook uses inline Python instead of relying on a local helper script.

In [ ]:
import argparse
import json
from pathlib import Path

import vec2text

alias = 'jxm/t5-base__llama-7b__one-million-instructions__emb'
dataset = 'python_code_alpaca'
num_samples = 10
batch_size = 1
use_less_data = 32
output_json = 'results/minimal_eval_lmi_10.json'

experiment, trainer = vec2text.analyze_utils.load_experiment_and_trainer_from_pretrained(
    alias,
    use_less_data=use_less_data,
)
trainer.model.use_frozen_embeddings_as_input = False
trainer.args.per_device_eval_batch_size = batch_size

available = sorted(trainer.eval_dataset.keys())
if dataset not in trainer.eval_dataset:
    raise KeyError(f"Dataset {dataset!r} not found. Available eval datasets: {available}")

eval_dataset = trainer.eval_dataset[dataset]
if 'frozen_embeddings' in eval_dataset.column_names:
    eval_dataset = eval_dataset.remove_columns('frozen_embeddings')
eval_dataset = eval_dataset.select(range(min(num_samples, len(eval_dataset))))

metrics = trainer.evaluate(eval_dataset=eval_dataset)
reduced_metrics = {
    'alias': alias,
    'dataset': dataset,
    'num_samples': len(eval_dataset),
    'available_eval_datasets': available,
    'eval_bleu_score': metrics.get('eval_bleu_score'),
    'eval_bleu_score_sem': metrics.get('eval_bleu_score_sem'),
    'eval_exact_match': metrics.get('eval_exact_match'),
    'eval_exact_match_sem': metrics.get('eval_exact_match_sem'),
    'eval_accuracy': metrics.get('eval_accuracy'),
    'eval_perplexity': metrics.get('eval_perplexity'),
    'eval_loss': metrics.get('eval_loss'),
    'eval_runtime': metrics.get('eval_runtime'),
    'eval_samples_per_second': metrics.get('eval_samples_per_second'),
}

output_path = Path(output_json)
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(reduced_metrics, indent=2), encoding='utf-8')

print(json.dumps(reduced_metrics, indent=2))


In [ ]:
import json
from pathlib import Path

for path in [Path('results/minimal_eval_lmi_10.json')]:
    if path.exists():
        print('\n===', path, '===')
        print(json.dumps(json.loads(path.read_text(encoding='utf-8')), indent=2))


## 8. Optional: larger eval

Run this only after the 10-sample eval succeeds.

In [ ]:
# Change these values only after the smaller run works.
# num_samples = 20
# batch_size = 2
# use_less_data = 64
